# Gross-Pitaevskii

The Bose Einstein condensation is descripted by the Gross-Pitaevskii equation. The trap in our case can be very well described by an harmonic oscillator potential.

We assume that all the atmos are in the condensate, $\Psi(\bar{r})$ is normalized: $\int d^3r|\Psi(\bar{r})|^2=1$

In harmonic oscilator units, the GP equation reads:

$[-\frac{1}{2}\bar{\nabla}^2+\frac{1}{2}r^2+4\pi \bar{a}_sN|\Psi(\bar{r})|^2]\Psi(\bar{r}) = \mu\Psi(\bar{r})$


And the energy per particle:

$e = \frac{E}{N}=\int d^3r[-\frac{1}{2}\bar{\nabla}^2+\frac{1}{2}r^2]\Psi(\bar{r})+2\pi \bar{a}_sN\int d^3r|\Psi(\bar{r})|^4$

In a Bose-Einstein condensation all the boson are in the ground state, so, the monoparticular wave function will be:

$\Psi=\frac{R}{r}Y_{00}$ -->normalized to 1

$Y_{00}$ is constant ($Y_{00}=\frac{1}{\sqrt{4\pi}}$) so the equation is:

$[-\frac{1}{2}\frac{d^2}{dr^2}+\frac{1}{2}r^2+4\pi \bar{a}_sN(\frac{R}{r})^2\frac{1}{4\pi}]R(\bar{r}) = \mu R(\bar{r})$

The radial part comes from solving the Schrödinger equation for the ground state with an harmonic potential-->$V(r)=\frac{1}{2}mw^2r^2$, which is needed to trap the paticles to the condensate.
The normalized radial part is:

$R_{00}=2(\frac{m\omega}{\pi\hbar})^{\frac{3}{4}}e^{-\frac{m\omega r^2}{2\hbar}}=2\alpha^{\frac{3}{2}}\pi^{-\frac{1}{4}}e^{-\frac{\alpha^2 r^2}{2}}$

The time operator is:

$U=e^{-\frac{itH}{\hbar}}\xrightarrow[it=\tau-->t = -i\tau]{}U=e^{-\frac{\tau H}{\hbar}}$

We are using imaginary time. The triall wave function that we choose at $\tau=0$ with non-zero overlap with the ground state is:

$\Psi(x,\tau=0)=\sum c_n\psi_n(x)$

Adding the time dependace:

$\Psi(x,\tau)=\sum c_n\psi_n(x)e^{-\frac{E_n\tau}{\hbar}}$

$\tau$ is an imaginary time which if we take it to infinity the wave function results:

$\Psi(x,\tau)= c_0\psi_0(x)e^{-\frac{E_0\tau}{\hbar}}$

The states different than $\phi_0$ decay faster and only survives the ground state which we want to find. 

We will track the evolution with:

$R(r,t+\Delta t)=R(r,t)-\Delta tHR(r,t)$ (essentially a Taylor expansion of the decay $e^{-H\tau}$)

So the idea is to compute the evolution of R(r,t) starting with our trial wave function and updating it with the time step and the eigenvalue of the hamiltonian of R(r,t) (which is the chemical potential). This will lead us to a lower energy wave function, however, we are taking a portion of R(r,t) so we will need to normalized it at each step.

The energy will be:

$e=\int dr[-\frac{1}{2}RR''+\frac{1}{2}R^2r^2]+\frac{\bar{a}_sN}{2}\int drr^2(\frac{R}{r})^4$

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [47]:
#aa = 10000 Number of atoms
#n1 number of integration steps
#step integration step
#a0 scattering lenght in h.o. function
#alpha the starting parameter for the h.o. function
#time the step in time
# itera number of iterations
a0,n1,step,aa,time,alpha,itera = tuple(map(float, input("Enter input values separated by space: ").split()))

Enter input values separated by space:  0 600 0.015 100000 0.0001 0.4 50000


In [48]:
#starting wave function

Y00 = np.sqrt(1/(4*np.pi)) #spherical harmonic with l=0
norm = 2*np.sqrt(alpha)**3/np.sqrt(np.sqrt(np.pi)) #starting normalization constant


xr = np.linspace(0, (n1-1)*step, int(n1)) #dimension of the system r
trialf = norm*xr * np.exp(-0.5 * alpha**2 * xr**2) #the value of our trial wave functiuon at each r

In [49]:
#hamiltionian for the chemical potential
def H(xr,trailf,step,a0,aa):
    kinetic = -0.5*np.divide(np.gradient(np.gradient(trialf,step),step),trialf,out=np.zeros_like(trialf), where=trialf!=0)
    pot = 0.5*xr**2
    inter = a0*aa*np.divide(trialf, xr, out=np.zeros_like(trialf), where=xr!=0)**2
    mu = kinetic+pot+inter
    return mu

In [50]:
#energy per particle
def e(trialf,step,xr,a0,aa):
    kinetic = -0.5*trialf*np.gradient(np.gradient(trialf,step),step)
    pot = 0.5*xr**2*trialf**2
    inter = 0.5*a0*aa*xr**2*np.divide(trialf, xr, out=np.zeros_like(trialf), where=xr!=0)**4
    ene = kinetic+pot+inter
    return np.sum(ene)*step

In [51]:
as3n=aa*a0*a0*a0 #Interaction strenght

In [53]:
#main loop to find the convergence
count = 0
for i in range(int(itera)):
    count += 1

    mu = H(xr,trialf,step,a0,aa)    #chemical potential of trailf
    ene = e(trialf,step,xr,a0,aa)   #energy of trailf
    mu[0] = 0                       #contour condition
    
    newf = trialf-time*mu*trialf                #evolution in time of trailf
    newnorm = np.sqrt(np.sum(newf**2) * step)   # new normalization constant

    trialf = newf/newnorm  #update to the new trialf
    
    if count == 200:
        print("ene0 = ",ene)
        count = 0
        
    if i == itera-1:
        print(ene,np.mean(mu))

ene0 =  1.4998593637528204
ene0 =  1.4998593635225526
ene0 =  1.4998593633099884
ene0 =  1.4998593631137662
ene0 =  1.4998593629326298
ene0 =  1.4998593627654184
ene0 =  1.499859362611064
ene0 =  1.4998593624685754
ene0 =  1.499859362337042
ene0 =  1.4998593622156207
ene0 =  1.4998593621035357
ene0 =  1.4998593620000669
ene0 =  1.499859361904553
ene0 =  1.4998593618163822
ene0 =  1.4998593617349902
ene0 =  1.4998593616598561
ene0 =  1.4998593615904976
ene0 =  1.4998593615264726
ene0 =  1.4998593614673688
ene0 =  1.49985936141281
ene0 =  1.4998593613624451
ene0 =  1.499859361315952
ene0 =  1.499859361273034
ene0 =  1.499859361233415
ene0 =  1.499859361196843
ene0 =  1.499859361163082
ene0 =  1.4998593611319166
ene0 =  1.4998593611031474
ene0 =  1.4998593610765898
ene0 =  1.499859361052074
ene0 =  1.4998593610294428
ene0 =  1.4998593610085522
ene0 =  1.4998593609892676
ene0 =  1.4998593609714648
ene0 =  1.4998593609550315
ene0 =  1.4998593609398614
ene0 =  1.4998593609258577
ene0 =  1.49